# Bài 7 · Đánh giá đáng tin cậy · Thực hành trên lớp (v3)

Mở notebook từ thư mục bài. Dữ liệu:
- `data/olist_model_table.csv`: bảng mô hình đóng băng của Bài 4–6 (79.686 đơn, ba tập theo thời gian);
- `data/olist_extended_v3.csv.gz`: bảng mở rộng (đơn duyệt 01/2017–03/2018, nhãn cuối cùng, kèm người bán, khách, đánh giá, ngày giao hãng).

Hai tệp được kiểm SHA-256 ở ô đầu. Mã cho sẵn; việc của bạn là chạy từng ô, đối chiếu với slide và trả lời câu hỏi **in đậm**. Ô có dấu ✅ là ô tự kiểm.

Notebook **không** đọc các đơn duyệt 01–03/2018: đó là tập cuối của project. Notebook cũng không tính gì trên test.

In [1]:
from pathlib import Path
import hashlib, warnings
from fractions import Fraction
from math import comb
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, cross_validate
from sklearn.metrics import average_precision_score, confusion_matrix
ROOT = Path.cwd()
sha = lambda p: hashlib.sha256((ROOT / p).read_bytes()).hexdigest()
assert sha('data/olist_model_table.csv') == 'cee5d8df21bb930d03c1130bff0478edc1934c5206030e9be6ec5223d1783825', 'sai bảng mô hình'
assert sha('data/olist_extended_v3.csv.gz') == 'ce93d20c8b46f984d607c79a2751cfb8c165285351544847d2a6e2e8e85fa9e3', 'sai bảng mở rộng'
DATES = ['order_approved_at', 'order_delivered_customer_date', 'order_estimated_delivery_date']
FEATURES = ['distance_km', 'promised_days', 'price', 'freight_value', 'n_items', 'weight_g']
frozen = pd.read_csv(ROOT / 'data/olist_model_table.csv', parse_dates=DATES)
ext = pd.read_csv(ROOT / 'data/olist_extended_v3.csv.gz', dtype={'period': str},
                  parse_dates=DATES + ['order_delivered_carrier_date', 'review_creation_date'])
ext = ext[ext.period == '2017']        # chỉ năm 2017: không đọc tập cuối của project
extra = ['order_id', 'seller_id', 'customer_unique_id', 'review_score', 'review_creation_date', 'order_delivered_carrier_date', 'carrier_days']
ext_all_tv = pd.read_csv(ROOT / 'data/olist_extended_v3.csv.gz', dtype={'period': str}, usecols=extra + ['split_bang_dong_bang'],
                         parse_dates=['review_creation_date', 'order_delivered_carrier_date'])
ext_all_tv = ext_all_tv[ext_all_tv.split_bang_dong_bang.isin(['train', 'validation'])].drop(columns='split_bang_dong_bang')
m = frozen[frozen.split != 'test'].merge(ext_all_tv, on='order_id', how='left', validate='one_to_one')
TR = m[m.split == 'train'].reset_index(drop=True); VA = m[m.split == 'validation'].reset_index(drop=True)
ytr, yva = TR.late.to_numpy(), VA.late.to_numpy()
def LOG():
    return Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler()), ('lr', LogisticRegression(C=1, solver='newton-cg', max_iter=100))])
def TREE(d):
    return Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler()), ('dt', DecisionTreeClassifier(max_depth=d, min_samples_leaf=200, random_state=42))])
ap = lambda y, s: float(average_precision_score(np.asarray(y).astype(int), s))
def kiem(ten, dk):
    assert dk, '❌ ' + ten
    print('✅', ten)
gan = lambda a, b: abs(a - b) < 1e-9
print('train', len(TR), 'đơn,', round(ytr.mean() * 100, 2), '% trễ · validation', len(VA), 'đơn,', round(yva.mean() * 100, 2), '% trễ')

train 29031 đơn, 3.24 % trễ · validation 10389 đơn, 6.1 % trễ


In [2]:
kiem('ba tập như Bài 4–6: train 29.031, validation 10.389', (len(TR), len(VA)) == (29031, 10389))
kiem('đủ cột người bán cho mọi đơn train', TR.seller_id.notna().all())

✅ ba tập như Bài 4–6: train 29.031, validation 10.389
✅ đủ cột người bán cho mọi đơn train


## Chặng 1 · Lúc dự đoán, ta được biết gì? (slide 5–17)
Bài toán: cảnh báo đơn trễ **ngay lúc duyệt**. Mô hình đã khoá: logistic của Bài 5 trên sáu đặc trưng.

In [3]:
gap = (VA.order_delivered_customer_date.dt.normalize() - VA.order_estimated_delivery_date.dt.normalize()).dt.days.to_numpy()
print('ma trận nhầm lẫn của quy tắc “giao − hẹn > 0”:', confusion_matrix(yva, (gap > 0).astype(int)).ravel(), '· AP của cột giao − hẹn:', round(ap(yva, gap), 3))
base = LOG().fit(TR[FEATURES], ytr); ap_base = ap(yva, base.predict_proba(VA[FEATURES])[:, 1])
F_rev = FEATURES + ['review_score']; ap_rev = ap(yva, LOG().fit(TR[F_rev], ytr).predict_proba(VA[F_rev])[:, 1])
F_car = FEATURES + ['carrier_days']; ap_car = ap(yva, LOG().fit(TR[F_car], ytr).predict_proba(VA[F_car])[:, 1])
print('AP validation · 6 đặc trưng:', round(ap_base, 3), '· + điểm đánh giá:', round(ap_rev, 3), '· + ngày giao hãng:', round(ap_car, 3))
print(TR.groupby('review_score').late.agg(['sum', 'size', 'mean']).round(4))

ma trận nhầm lẫn của quy tắc “giao − hẹn > 0”: [9755    0    0  634] · AP của cột giao − hẹn: 1.0
AP validation · 6 đặc trưng: 0.116 · + điểm đánh giá: 0.337 · + ngày giao hãng: 0.285
              sum   size    mean
review_score                    
1.0           431   2147  0.2007
2.0            83    840  0.0988
3.0            97   2377  0.0408
4.0           109   5864  0.0186
5.0           189  17596  0.0107


In [4]:
kiem('quy tắc ngày giao tái dựng nhãn: FP = FN = 0 (slide 9)', confusion_matrix(yva, (gap > 0).astype(int)).ravel()[1:3].sum() == 0)
kiem('logistic Bài 5 trên validation: 0,1165 (slide 11)', gan(ap_base, 0.11646952008298003))
kiem('thêm điểm đánh giá: 0,337', gan(ap_rev, 0.33720468650328433))
kiem('thêm số ngày đến lúc giao hãng: 0,285 (slide 12)', gan(ap_car, 0.2853014314449483))

✅ quy tắc ngày giao tái dựng nhãn: FP = FN = 0 (slide 9)
✅ logistic Bài 5 trên validation: 0,1165 (slide 11)
✅ thêm điểm đánh giá: 0,337
✅ thêm số ngày đến lúc giao hãng: 0,285 (slide 12)


**Câu hỏi.**
1. Cột nào trong ba cột vừa thêm hợp lệ nếu dự đoán **lúc giao hãng**? Cột nào vẫn là rò rỉ?
2. Điểm đánh giá không nằm trong công thức của nhãn. Vì sao nó vẫn là rò rỉ khi dự đoán lúc duyệt? Dùng câu 2 và câu 4 của bảng kiểm.
3. Tìm trong `TR` số đơn có ngày tạo đánh giá **trước** ngày giao thật. Tỷ lệ trễ của nhóm này là bao nhiêu, và giải thích thế nào?

## Chặng 2 · Học từ đâu, áp cho ai? (slide 18–31)

In [5]:
# 2a · chuẩn hoá bằng tay (mô phỏng): fit trên 4 đơn học, transform đơn mới
x = [12, 18, 4, 10]; moi = 20
mu = Fraction(sum(x), 4); var = sum((Fraction(v) - mu) ** 2 for v in x) / 4
print('μ =', mu, '· σ² =', var, '· z của đơn mới =', (moi - mu) / 5)
print('StandardScaler cho cùng kết quả:', StandardScaler().fit(np.array(x, float)[:, None]).transform([[moi]]).ravel())
# 2b · vòng K-fold tự viết so với cross_validate(Pipeline)
skf = StratifiedKFold(5, shuffle=True, random_state=7); folds = list(skf.split(TR[FEATURES], ytr))
lib = cross_validate(LOG(), TR[FEATURES], ytr, cv=skf, scoring='average_precision')['test_score']
own = [ap(ytr[b], LOG().fit(TR[FEATURES].iloc[a], ytr[a]).predict_proba(TR[FEATURES].iloc[b])[:, 1]) for a, b in folds]
print('tự viết:', np.round(own, 4), '· thư viện:', np.round(lib, 4))
# 2c · chuẩn hoá ngoài fold
pre = Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]).fit(TR[FEATURES]); Z = pre.transform(TR[FEATURES])
out_sc = [ap(ytr[b], LogisticRegression(C=1, solver='newton-cg', max_iter=100).fit(Z[a], ytr[a]).predict_proba(Z[b])[:, 1]) for a, b in folds]
print('chênh lớn nhất khi chuẩn hoá ngoài fold:', max(abs(p - q) for p, q in zip(own, out_sc)))

μ = 11 · σ² = 25 · z của đơn mới = 9/5
StandardScaler cho cùng kết quả: [1.8]


tự viết: [0.0505 0.0443 0.0661 0.0605 0.0686] · thư viện: [0.0505 0.0443 0.0661 0.0605 0.0686]
chênh lớn nhất khi chuẩn hoá ngoài fold: 2.6033154792506896e-05


In [6]:
kiem('μ = 11, σ² = 25, z = 9/5 (slide 19)', mu == 11 and var == 25 and (moi - mu) / 5 == Fraction(9, 5))
kiem('vòng tự viết trùng cross_validate (slide 25)', max(abs(p - q) for p, q in zip(own, lib)) < 1e-12)
kiem('AP fold 1 = 0,0505', gan(own[0], 0.050516977156266296))
kiem('chuẩn hoá ngoài fold chỉ lệch cỡ 10⁻⁵ (slide 26)', gan(max(abs(p - q) for p, q in zip(own, out_sc)), 2.6033154792506896e-05))

✅ μ = 11, σ² = 25, z = 9/5 (slide 19)
✅ vòng tự viết trùng cross_validate (slide 25)
✅ AP fold 1 = 0,0505
✅ chuẩn hoá ngoài fold chỉ lệch cỡ 10⁻⁵ (slide 26)


In [7]:
# 2d · mã hoá người bán: r = (số trễ + m·p̄)/(số đơn + m), m = 10
def fit_rate(seller, y, m=10):
    df = pd.DataFrame({'s': np.asarray(seller), 'y': np.asarray(y).astype(int)})
    p0 = float(df.y.mean()); g = df.groupby('s').y.agg(['sum', 'size'])
    return ((g['sum'] + m * p0) / (g['size'] + m)).to_dict(), p0
def apply_rate(seller, fitted):
    rate, p0 = fitted
    return np.array([rate.get(s, p0) for s in seller], float)
def cross_fit_rate(seller, y, m=10):
    # r của CHÍNH các dòng học, tính chéo 5 phần (như TargetEncoder.fit_transform): không chứa nhãn của chính dòng đó
    seller = np.asarray(seller); y = np.asarray(y).astype(int); out = np.empty(len(y))
    for a_, b_ in StratifiedKFold(5, shuffle=True, random_state=7).split(np.zeros(len(y)), y):
        out[b_] = apply_rate(seller[b_], fit_rate(seller[a_], y[a_], m))
    return out
s_tr = TR.seller_id.to_numpy(); r_all = apply_rate(s_tr, fit_rate(s_tr, ytr))
cv_out, cv_in, cv_cf = [], [], []
for a, b in folds:
    Xa, Xb = TR[FEATURES].iloc[a].assign(r=r_all[a]), TR[FEATURES].iloc[b].assign(r=r_all[b])
    cv_out.append(ap(ytr[b], LOG().fit(Xa, ytr[a]).predict_proba(Xb)[:, 1]))          # SAI: r tính trước khi chia
    f = fit_rate(s_tr[a], ytr[a]); Xb = TR[FEATURES].iloc[b].assign(r=apply_rate(s_tr[b], f))
    Xa = TR[FEATURES].iloc[a].assign(r=apply_rate(s_tr[a], f))
    cv_in.append(ap(ytr[b], LOG().fit(Xa, ytr[a]).predict_proba(Xb)[:, 1]))           # LỖI TINH HƠN: r của dòng học tính trong mẫu
    Xa = TR[FEATURES].iloc[a].assign(r=cross_fit_rate(s_tr[a], ytr[a]))
    cv_cf.append(ap(ytr[b], LOG().fit(Xa, ytr[a]).predict_proba(Xb)[:, 1]))           # ĐÚNG: trong fold, dòng học tính chéo
f_tr = fit_rate(s_tr, ytr); Xva_r = VA[FEATURES].assign(r=apply_rate(VA.seller_id, f_tr))
m_in = LOG().fit(TR[FEATURES].assign(r=apply_rate(s_tr, f_tr)), ytr); m_cf = LOG().fit(TR[FEATURES].assign(r=cross_fit_rate(s_tr, ytr)), ytr)
ap_val_in, ap_val_cf = ap(yva, m_in.predict_proba(Xva_r)[:, 1]), ap(yva, m_cf.predict_proba(Xva_r)[:, 1])
print('CV · không r:', round(np.mean(own), 4), '· r ngoài fold:', round(np.mean(cv_out), 4), '· trong fold, dòng học trong mẫu:', round(np.mean(cv_in), 4), '· trong fold, tính chéo:', round(np.mean(cv_cf), 4))
print('validation · không r:', round(ap_base, 4), '· r tính chéo:', round(ap_val_cf, 4), '· r dòng học trong mẫu:', round(ap_val_in, 4))
print('hệ số của r (sau chuẩn hoá) · trong mẫu:', round(m_in.named_steps['lr'].coef_[0][-1], 2), '· tính chéo:', round(m_cf.named_steps['lr'].coef_[0][-1], 2))

CV · không r: 0.058 · r ngoài fold: 0.1692 · trong fold, dòng học trong mẫu: 0.0717 · trong fold, tính chéo: 0.075
validation · không r: 0.1165 · r tính chéo: 0.112 · r dòng học trong mẫu: 0.0943
hệ số của r (sau chuẩn hoá) · trong mẫu: 0.65 · tính chéo: 0.26


In [8]:
kiem('r ngoài fold: 0,169; trong fold, tính chéo: 0,075 (slide 27)', gan(np.mean(cv_out), 0.16917905051817214) and gan(np.mean(cv_cf), 0.07504288077103988))
kiem('tháng sau: r tính chéo 0,112, không hơn mốc 0,116', gan(ap_val_cf, 0.11199694321316733) and ap_val_cf < ap_base)
kiem('lỗi tinh hơn: r của dòng học tính trong mẫu, tháng sau chỉ còn 0,094', gan(ap_val_in, 0.09431796243272036) and gan(np.mean(cv_in), 0.07173038579690497))

✅ r ngoài fold: 0,169; trong fold, tính chéo: 0,075 (slide 27)
✅ tháng sau: r tính chéo 0,112, không hơn mốc 0,116
✅ lỗi tinh hơn: r của dòng học tính trong mẫu, tháng sau chỉ còn 0,094


**Câu hỏi.**
1. Với người bán chỉ có một đơn trong train, `r_all` của đơn đó nhận mấy giá trị? Viết công thức theo nhãn của đơn. Với `cross_fit_rate` thì sao?
2. Vì sao tính r của dòng học trong mẫu làm tháng sau giảm nhiều, còn tính chéo thì chỉ giảm rất ít so với mốc không r? Dùng hai hệ số của r in ở trên để giải thích.

## Chặng 3 · Chia theo đối tượng sẽ gặp khi triển khai (slide 33–44)

In [9]:
print((ext.groupby('month').late.mean() * 100).round(1).to_string())
DV = pd.concat([TR, VA]).reset_index(drop=True); ydv = DV.late.to_numpy()
skf2 = StratifiedKFold(5, shuffle=True, random_state=7)
rl_f = cross_validate(LOG(), DV[FEATURES], ydv, cv=skf2, scoring='average_precision')['test_score']
rt_f = cross_validate(TREE(6), DV[FEATURES], ydv, cv=skf2, scoring='average_precision')['test_score']
rl, rt, cay_hon = rl_f.mean(), rt_f.mean(), int((rt_f > rl_f).sum())
ft = ap(yva, TREE(6).fit(TR[FEATURES], ytr).predict_proba(VA[FEATURES])[:, 1])
print('CV ngẫu nhiên · logistic', round(rl, 4), '· cây sâu 6', round(rt, 4), '· cây hơn ở', cay_hon, '/ 5 fold')
print('train → validation · logistic', round(ap_base, 4), '· cây sâu 6', round(ft, 4))
okt = ext[ext.month == '2017-10']
print('T10/2017 · mọi đơn:', round(okt.late.mean() * 100, 2), '% · chỉ đơn đã giao trước 01/11:', round(okt[okt.order_delivered_customer_date < '2017-11-01'].late.mean() * 100, 2), '%')

month
2017-01     2.9
2017-02     2.8
2017-03     4.6
2017-04     6.5
2017-05     3.0
2017-06     3.0
2017-07     2.8
2017-08     2.9
2017-09     4.4
2017-10     4.1
2017-11    12.2
2017-12     7.9


CV ngẫu nhiên · logistic 0.0738 · cây sâu 6 0.0746 · cây hơn ở 3 / 5 fold
train → validation · logistic 0.1165 · cây sâu 6 0.1005
T10/2017 · mọi đơn: 4.14 % · chỉ đơn đã giao trước 01/11: 0.67 %


In [10]:
kiem('CV ngẫu nhiên: gần như ngang (cây hơn 3/5 fold); chấm tháng sau: logistic hơn rõ (slide 35)', cay_hon == 3 and abs(rt - rl) < 0.001 and ap_base - ft > 0.01 and gan(rl, 0.07382743187817599) and gan(rt, 0.07459889234788035))
kiem('nhãn sẵn có: 4,14% so với 0,67% (slide 36)', gan(okt.late.mean(), 0.041432109885161))

✅ CV ngẫu nhiên: gần như ngang (cây hơn 3/5 fold); chấm tháng sau: logistic hơn rõ (slide 35)
✅ nhãn sẵn có: 4,14% so với 0,67% (slide 36)


In [11]:
# 3b · đánh giá cuốn theo tháng: học các đơn đã giao trước ngày 1 của tháng m, chấm mọi đơn duyệt trong tháng m
rows = []
for mo in ['2017-05', '2017-06', '2017-07', '2017-08', '2017-09', '2017-10', '2017-11', '2017-12']:
    hoc = ext[ext.order_delivered_customer_date < pd.Timestamp(mo + '-01')]; cham = ext[ext.month == mo]
    s = {k: ap(cham.late, mdl.fit(hoc[FEATURES], hoc.late).predict_proba(cham[FEATURES])[:, 1]) for k, mdl in [('sau6', TREE(6)), ('sau7', TREE(7)), ('log', LOG())]}
    rows.append(dict(thang=mo, n_hoc=len(hoc), n_cham=len(cham), ty_le=round(cham.late.mean(), 4), **{k: round(v, 4) for k, v in s.items()}, d76=s['sau7'] - s['sau6']))
cuon = pd.DataFrame(rows); print(cuon.drop(columns='d76').to_string(index=False))

  thang  n_hoc  n_cham  ty_le   sau6   sau7    log
2017-05   5850    3547 0.0304 0.0288 0.0293 0.0545
2017-06   9601    3143 0.0302 0.0389 0.0369 0.0419
2017-07  12824    3827 0.0280 0.0409 0.0389 0.0491
2017-08  16278    4217 0.0287 0.0330 0.0324 0.0403
2017-09  20580    4162 0.0440 0.0627 0.0656 0.0859
2017-10  24537    4441 0.0414 0.0566 0.0572 0.0714
2017-11  29031    7149 0.1223 0.1613 0.1656 0.1807
2017-12  33701    5675 0.0786 0.0849 0.0884 0.0997


In [12]:
kiem('phần học fold T11 = train của Bài 4–6 (29.031 đơn) (slide 37)', int(cuon.n_hoc[6]) == 29031)
kiem('hiệu sâu 7 − sâu 6 khớp từng tháng', max(abs(a - b) for a, b in zip(cuon.d76, [0.0005866667044098738, -0.0020753113419229308, -0.0019757138263560847, -0.0005820612048336032, 0.0029542372006282763, 0.0006211671996246559, 0.004358834703454967, 0.0035278768656470116])) < 1e-9)

✅ phần học fold T11 = train của Bài 4–6 (29.031 đơn) (slide 37)
✅ hiệu sâu 7 − sâu 6 khớp từng tháng


In [13]:
# 3c · chia theo người bán (StratifiedGroupKFold): r có còn giúp không?
sgk = StratifiedGroupKFold(5, shuffle=True, random_state=7); g_cf, g_base = [], []
for a, b in sgk.split(TR[FEATURES], ytr, groups=s_tr):
    assert not set(s_tr[a]) & set(s_tr[b])
    f = fit_rate(s_tr[a], ytr[a]); Xb = TR[FEATURES].iloc[b].assign(r=apply_rate(s_tr[b], f))
    g_cf.append(ap(ytr[b], LOG().fit(TR[FEATURES].iloc[a].assign(r=cross_fit_rate(s_tr[a], ytr[a])), ytr[a]).predict_proba(Xb)[:, 1]))
    g_base.append(ap(ytr[b], LOG().fit(TR[FEATURES].iloc[a], ytr[a]).predict_proba(TR[FEATURES].iloc[b])[:, 1]))
print('chia theo người bán · không r', round(np.mean(g_base), 4), '· có r (dòng học tính chéo)', round(np.mean(g_cf), 4))

chia theo người bán · không r 0.0558 · có r (dòng học tính chéo) 0.0565


In [14]:
kiem('chia theo người bán: 0,0558 → 0,0565 (slide 40)', gan(np.mean(g_base), 0.0557888797948799) and gan(np.mean(g_cf), 0.05648114652052272))

✅ chia theo người bán: 0,0558 → 0,0565 (slide 40)


**Câu hỏi.**
1. Một đơn duyệt 14/07/2017, giao 02/08/2017: nó nằm trong phần học của những fold nào trong bảng `cuon`, và được chấm ở fold nào? Kiểm bằng cách lọc `ext` theo đúng quy tắc của ô 3b.
2. Vì sao chuyển sang chia theo người bán làm AP của mô hình có r giảm từ khoảng 0,075 xuống 0,0565, còn chia theo khách gần như không đổi AP?

## Chặng 4 · Chọn mô hình mà không tự lừa mình (slide 45–56)

In [15]:
def kiem_dinh_dau(thang, n):
    k = max(thang, n - thang)
    return min(Fraction(1), 2 * Fraction(sum(comb(n, j) for j in range(k, n + 1)), 2 ** n))
w7, w6 = int((cuon.d76 > 0).sum()), int((cuon.d76 < 0).sum()); p = kiem_dinh_dau(w7, w7 + w6)
ket_luan = 'sâu 7 tốt hơn' if p < 0.05 and w7 > w6 else ('sâu 6 tốt hơn' if p < 0.05 else 'chưa phân biệt được')
print('sâu 7 thắng', w7, 'thua', w6, '· p =', p, '≈', round(float(p), 3), '→', ket_luan)
print('logistic hơn cây sâu 6 ở', int((cuon.log > cuon.sau6).sum()), '/ 8 tháng (quan sát mô tả, không ghi trước)')
def E_max(n, K):          # kỳ vọng số đúng của bộ cao nhất trong K bộ đoán ngẫu nhiên, n nhãn
    pmf = [Fraction(comb(n, k), 2 ** n) for k in range(n + 1)]; cdf = [sum(pmf[:k + 1]) for k in range(n + 1)]
    return sum(1 - cdf[k - 1] ** K for k in range(1, n + 1))
print('n = 2: K = 2 →', E_max(2, 2), '; K = 3 →', E_max(2, 3), '· n = 50, K = 20 → tỷ lệ đúng', round(float(E_max(50, 20)) / 50, 4))

sâu 7 thắng 5 thua 3 · p = 93/128 ≈ 0.727 → chưa phân biệt được
logistic hơn cây sâu 6 ở 8 / 8 tháng (quan sát mô tả, không ghi trước)
n = 2: K = 2 → 11/8 ; K = 3 → 25/16 · n = 50, K = 20 → tỷ lệ đúng 0.6313


In [16]:
kiem('sâu 7 thắng 5, thua 3, p = 93/128: chưa phân biệt được (slide 50)', (w7, w6, p, ket_luan) == (5, 3, Fraction(93, 128), 'chưa phân biệt được'))
kiem('E[max] = 11/8 và 25/16 (slide 48)', E_max(2, 2) == Fraction(11, 8) and E_max(2, 3) == Fraction(25, 16))
kiem('20 bộ đoán ngẫu nhiên: 63,1% (slide 47)', gan(float(E_max(50, 20)) / 50, 0.6313052817960847))

✅ sâu 7 thắng 5, thua 3, p = 93/128: chưa phân biệt được (slide 50)
✅ E[max] = 11/8 và 25/16 (slide 48)
✅ 20 bộ đoán ngẫu nhiên: 63,1% (slide 47)


**Câu hỏi.**
1. Với 12 tháng không hoà, một mô hình phải thắng ít nhất bao nhiêu tháng thì p < 0,05? Tính bằng `kiem_dinh_dau`.
2. Viết một câu báo cáo theo mẫu có điều kiện (slide 54) cho phép so logistic với cây sâu 6, nói rõ vì sao đó chỉ là quan sát mô tả.
3. Khoảng bootstrap của Bài 6 không chứa 0. Vì sao nó không mâu thuẫn với kết luận “chưa phân biệt được” ở trên?